# 01. Extracción de datos 

Se extraen los datos de la API Tenrai.org para construir DataFrames con la información requerida 

In [1]:
#%pip install pandas requests ipywidgets

In [27]:
import json
import math
import os
import re
import threading
import time
from collections import deque
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

In [28]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yaml").exists())
CHECKPOINTS = ROOT / "checkpoints"
DATA = ROOT / "data"
CHECKPOINTS.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

CHPT_TOP = CHECKPOINTS / "top_checkpoint.json"
CHPT_EXPANSION = CHECKPOINTS / "relations_checkpoint.json"

RPS = 3
RPM = 90
BASE_URL = "https://api.tenrai.org/v1"

**Rate Limiter** 

Es necesario esperar un tiempo entre cada llamada a la API para evitar que bloqueen el acceso. 

In [29]:
class RateLimiter:
    """Espacia las llamadas a la API de forma uniforme, respetando límites por segundo y por minuto."""
    def __init__(self, rps=2, rpm=90):
        self.rps = rps # Maximo de llamadas por segundo
        self.rpm = rpm # Maximo de llamadas por minuto
        self.interval = 1.0 / rps  # Intervalo de llamadas por segundo
        self.timestamps = deque() # Almacena los timestamps de las llamadas recientes
        self.last_call = None # Almacena el timestamp de la última llamada
        self.lock = threading.Lock() # Bloqueo para asegurar que las llamadas sean thread-safe

    def waits(self): 
        with self.lock: 
            # Timestamp actual
            now = time.monotonic()

            # Reinicia si pasa un minuto
            while self.timestamps and now - self.timestamps[0] > 60:
                self.timestamps.popleft()

            # Espera si supera las llamadas por minuto
            if len(self.timestamps) >= self.rpm:
                wait = 60 - (now - self.timestamps[0])
                if wait > 0:
                    time.sleep(wait)
                now = time.monotonic()  # recalcular timestamp tras dormir

            # Intervalos entre varias llamadas en un segundo
            if self.last_call is not None:
                waited = now - self.last_call
                # Completa el intervalo antes de hacer otra llamada
                if waited < self.interval:
                    time.sleep(self.interval - waited)
                    now = time.monotonic()  # recalcular timestamp tras dormir

            self.last_call = now
            self.timestamps.append(now)

LIMITER = RateLimiter(RPS, RPM)
LIMITER.waits()            

**Requests y Checkpoints** 

Funciones para poder hacer las solicitudes a la API y para guardar los datos en formato JSON durante el proceso (checkpoints)

In [31]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "anime-db-project/1.0"})

def save_json(path, obj):
    """Guarda los datos en un json temporal. Luego reemplaza el archivo original para evitar corrupciones si se interrumpe la escritura.
    path: ruta del archivo a guardar
    obj: objeto a guardar (dict, list, etc.)"""
    tmp = Path(path).with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)
    os.replace(tmp, path)

def get_json(path, params=None, limiter=LIMITER, retries=3, timeout=15):
    """
    GET a la API.
    path: ruta del archivo a guardar 
    params: Parametros para realizar el request
    limiter: Limitador para el número de requests
    retries: Número de reintentos por request
    timeout: Tiempo que espera por la respuesta de la API
    Devuelve:
      - dict con el JSON en caso de éxito
      - {"data": [], "not_found": True} si el recurso no existe (404)
      - None si falló tras todos los reintentos (error de red / 5xx / 429 persistente)
    """
    url = f"{BASE_URL}/{path}"
    for attempt in range(1, retries + 1):
        # Espera antes de realizar el request 
        limiter.waits()
        try:
            # Hace el GET a la API
            r = SESSION.get(url, params=params, timeout=timeout)
            # Error 404 si no logra encontrar lo que esta buscando
            if r.status_code == 404:
                return {"data": [], "not_found": True}
            # Error 429 si se han enviado demasiadas peticiones
            if r.status_code == 429:
                try:
                    wait = float(r.headers.get("Retry-After", 5))
                except ValueError:
                    wait = 5
                time.sleep(wait)
                continue
            # Error 500 o similar indica que hay un error en el servidor 
            if r.status_code >= 500:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            return r.json()
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"Error en {url}: {e}. Intento {attempt}/{retries}")
            time.sleep(2 ** attempt)
    return None

get_json("ping")  # Test de conexión a la API

{'data': [], 'not_found': True}

**Parsear resultados** 

Funciones para recuperar los datos extraidos de las llamadas a la API y obtener los campos que se buscan agregar a la base de datos

In [32]:

def _items(lst):
    """Extrae IDs y nombres de animes en una lista 
    [{mal_id, name, ...}] -> [{mal_id, name}]"""
    return [{"mal_id": x["mal_id"], "name": x.get("name")} for x in (lst or []) if x.get("mal_id") is not None]


def parse_duration(text):
    """Convierte la duración de los anime a un formato númerico. 
    '24 min per ep' -> 24 ; '1 hr 55 min' -> 115 ; 'Unknown' -> None"""
    if not text:
        return None
    h = re.search(r"(\d+)\s*hr", text)
    m = re.search(r"(\d+)\s*min", text)
    total = (int(h.group(1)) * 60 if h else 0) + (int(m.group(1)) if m else 0)
    return total or None


def parse_anime(a):
    """Extrae los datos que se buscan de un anime: mal_id, mal_url, title, title_english, 
    title_japanese, type, source, status, episodes, duration, duration_min, rating, 
    score, scored_by, score_rank, popularity_rank, members, favorites, year, season, 
    aired_from, synopsis, background, image_url, producers, licensors, studios, 
    genres, explicit_genres, demographies, themes"""
    aired_from = ((a.get("aired") or {}).get("from") or "")[:10] or None   # 'YYYY-MM-DD'
    year = a.get("year")
    if year is None and aired_from:
        year = int(aired_from[:4])
    return {
        "mal_id": a["mal_id"],
        "mal_url": a.get("url"),
        "title": a.get("title"),
        "title_english": a.get("title_english"),
        "title_japanese": a.get("title_japanese"),
        "type": a.get("type"),
        "source": a.get("source"),
        "status": a.get("status"),
        "episodes": a.get("episodes"),
        "duration": a.get("duration"),
        "duration_min": parse_duration(a.get("duration")),
        "rating": a.get("rating"),
        "score": a.get("score"),
        "scored_by": a.get("scored_by"),
        "score_rank": a.get("rank"),
        "popularity_rank": a.get("popularity"),
        "members": a.get("members"),
        "favorites": a.get("favorites"),
        "year": year,
        "season": a.get("season"),
        "aired_from": aired_from,
        "synopsis": a.get("synopsis"),
        "background": a.get("background"),
        "image_url": ((a.get("images") or {}).get("jpg") or {}).get("image_url"),
        "producers": _items(a.get("producers")),
        "licensors": _items(a.get("licensors")),
        "studios": _items(a.get("studios")),
        "genres": _items(a.get("genres")),
        "explicit_genres": _items(a.get("explicit_genres")),
        "demographics": _items(a.get("demographics")),
        "themes": _items(a.get("themes")),
    }


def parse_relations(raw):
    """
    Se recorren las relaciones que estan en registradas para un anime, solo se guardan las que sean de tipo anime
    """
    out = []
    for r in raw or []:
        for e in r.get("entry", []):
            if e.get("type") == "anime":
                out.append({"mal_id": e["mal_id"], "relation": r.get("relation"), "name": e.get("name")})
    return out



In [33]:
TIPOS = {"tv", "movie", "ova", "special", "ona", "music", "cm", "pv", "tv_special"}
FILTROS_TOP = {"airing", "upcoming", "bypopularity", "favorite"}
# Prefijo con el que MAL escribe cada clasificación ("G - All Ages", "R - 17+ (violence & profanity)", "R+ - Mild Nudity"...)
RATING_PREFIJO = {"g": "G -", "pg": "PG -", "pg13": "PG-13", "r17": "R - 17+", "r": "R+", "rx": "Rx"}
FILTRO_STATUS = {"airing": "Currently Airing", "upcoming": "Not yet aired"}
FILTROS_DEFAULT = {"type": None, "filter": None, "rating": None, "sfw": False}


def check_filters(type=None, filter=None, rating=None):
    """Falla pronto (y con un mensaje claro) si un valor no existe, en vez de descubrirlo tras horas de extracción."""
    for nombre, valor, validos in (("type", type, TIPOS), ("filter", filter, FILTROS_TOP), ("rating", rating, set(RATING_PREFIJO))):
        if valor is not None and valor not in validos:
            raise ValueError(f"{nombre}={valor} no es válido. Opciones: {validos}")


def rating_code(texto):
    """'PG-13 - Teens 13 or older' -> 'pg13' ; None si no se reconoce."""
    for codigo, prefijo in RATING_PREFIJO.items():
        if texto and texto.startswith(prefijo):
            return codigo
    return None


def meets_filters(anime, type=None, filter=None, rating=None, sfw=False):
    """True si el anime (ya parseado con parse_anime) cumple TODOS los filtros indicados."""
    if type is not None and (anime.get("type") or "").lower().replace(" ", "_") != type:
        return False
    if rating is not None and rating_code(anime.get("rating")) != rating:
        return False
    if filter in FILTRO_STATUS and anime.get("status") != FILTRO_STATUS[filter]:
        return False
    if sfw and (rating_code(anime.get("rating")) == "rx" or anime.get("explicit_genres")):
        return False
    return True

**Obtener el Top de Animes**

Se recorre el top N de animes en MyAnimeList y utilizando las funciones anteriores se obtienen los datos que buscamos de los animes incluidos en la lista

In [34]:
def get_top_anime(amount=1000, per_page=25, retries=3, type=None, filter=None, rating=None, sfw=False):
    """
    Obtiene los animes más populares de la API de Tenrai.
    amount:   cantidad total de animes a obtener
    per_page: animes por página (máx. 25)
    type:     tv, movie, ova, special, ona, music, cm, pv, tv_special
    filter:   airing, upcoming, bypopularity, favorite (por defecto: ranking por score)
    rating:   clasificación de los animes (g, pg, pg-13, r-17, r, rx)
    sfw:      True para excluir contenido explícito
    Reanuda desde checkpoints/top_checkpoint.json si usa los mismos parámetros.
    """
    check_filters(type, filter, rating)
    params_id = {"type": type, "filter": filter, "rating": rating, "sfw": sfw, "per_page": per_page}
    results, last_page = [], 0
    if CHPT_TOP.exists():
        saved = json.loads(CHPT_TOP.read_text(encoding="utf-8"))
        if saved.get("params") == params_id:
            results, last_page = saved["anime_list"], saved["last_page"]
            print(f"Reanudando top: {len(results)} animes, última página {last_page}")
        else:
            print("El checkpoint del top usa otros parámetros: se ignora (bórralo si quieres empezar de cero).")

    seen = {a["mal_id"] for a in results}
    pages = math.ceil(amount / per_page)
    for page in tqdm(range(last_page + 1, pages + 1), desc="Top anime"):
        params = {"limit": per_page, "page": page, "type": type, "filter": filter}   # los None se omiten solos
        if sfw:
            params["sfw"] = "true"
        data = get_json("top/anime", params=params, retries=retries)
        if data is None:
            print(f"No se pudo descargar la página {page}; vuelve a ejecutar para reanudar.")
            break
        for anime in data.get("data", []):
            if anime["mal_id"] not in seen:
                seen.add(anime["mal_id"])
                results.append(parse_anime(anime))
        last_page = page
        save_json(CHPT_TOP, {"params": params_id, "last_page": last_page, "anime_list": results})
        if not data.get("pagination", {}).get("has_next_page", False):
            print("No hay más páginas disponibles.")
            break
    return results[:amount]

get_top_anime(100, per_page=25, type=None, filter=None, sfw=False)  # Test de extracción de top animes

El checkpoint del top usa otros parámetros: se ignora (bórralo si quieres empezar de cero).


Top anime:   0%|          | 0/4 [00:00<?, ?it/s]

[{'mal_id': 52991,
  'mal_url': 'https://myanimelist.net/anime/52991/Sousou_no_Frieren',
  'title': 'Sousou no Frieren',
  'title_english': "Frieren: Beyond Journey's End",
  'title_japanese': '葬送のフリーレン',
  'type': 'TV',
  'source': 'Manga',
  'status': 'Finished Airing',
  'episodes': 28,
  'duration': '24 min per ep',
  'duration_min': 24,
  'rating': 'PG-13 - Teens 13 or older',
  'score': 9.24,
  'scored_by': 951926,
  'score_rank': 1,
  'popularity_rank': 95,
  'members': 1542421,
  'favorites': 94716,
  'year': 2023,
  'season': 'fall',
  'aired_from': '2023-09-29',
  'synopsis': 'During their decade-long quest to defeat the Demon King, the members of the hero\'s party—Himmel himself, the priest Heiter, the dwarf warrior Eisen, and the elven mage Frieren—forge bonds through adventures and battles, creating unforgettable precious memories for most of them.\n\nHowever, the time that Frieren spends with her comrades is equivalent to merely a fraction of her life, which has lasted ov

**Extraer datos adicionales**

Funciones adicionales para obtener los datos sobre relaciones, recomendaciones y staff de los animes en la lista

In [35]:
def get_anime_relations(mal_id, retries=3):
    """Regresa las relaciones de un anime (solo anime), tal como aparecen en su página de MAL."""
    data = get_json(f"anime/{mal_id}/relations", retries=retries)
    if data is None:
        return None
    return parse_relations(data.get("data"))


def get_anime_full(mal_id, retries=3):
    """
    Regresa: dict con el los datos completos de un anime 
    | {} si no existe (404) | None si hubo error de red (reintentar luego).
    `/full` ya trae las relaciones; solo si faltan se pide `/relations`.
    """
    data = get_json(f"anime/{mal_id}/full", retries=retries)
    if data is None:
        return None
    raw = data.get("data")
    if not raw:
        return {}
    anime = parse_anime(raw)
    if "relations" in raw:
        anime["relations"] = parse_relations(raw["relations"])
    else:
        anime["relations"] = get_anime_relations(mal_id, retries) or []
    return anime


def get_anime_staff(mal_id, retries=3):
    """Regresa una lista con las personas en el staff de un anime
    [{person_id, name, positions: [...]}] | None si hubo error."""
    data = get_json(f"anime/{mal_id}/staff", retries=retries)
    if data is None:
        return None
    return [
        {"person_id": s["person"]["mal_id"], "name": s["person"].get("name"), "positions": s.get("positions") or []}
        for s in data.get("data", [])
        if (s.get("person") or {}).get("mal_id") is not None
    ]


def get_anime_recommendations(mal_id, max_recs=20, retries=3):
    """[{mal_id, title, votes}] con las `max_recs` más votadas | None si hubo error."""
    data = get_json(f"anime/{mal_id}/recommendations", retries=retries)
    if data is None:
        return None
    recs = [
        {"mal_id": r["entry"]["mal_id"], "title": r["entry"].get("title"), "votes": r.get("votes") or 0}
        for r in data.get("data", [])
        if (r.get("entry") or {}).get("mal_id") is not None
    ]
    recs.sort(key=lambda r: r["votes"], reverse=True)
    return recs[:max_recs]

get_anime_relations(1)  # Test de extracción de relaciones

[{'mal_id': 5,
  'relation': 'Side Story',
  'name': 'Cowboy Bebop: Tengoku no Tobira'},
 {'mal_id': 17205,
  'relation': 'Side Story',
  'name': 'Cowboy Bebop: Ein no Natsuyasumi'},
 {'mal_id': 4037,
  'relation': 'Summary',
  'name': 'Cowboy Bebop: Yose Atsume Blues'}]

**Extraer relaciones**

Función que extrae datos adicionales de animes con relaciones hacia los animes en el Top (precuelas, secuelas, recomendaciones, etc.). Esto se realiza de tal forma que se complete la lista con los animes que forman parte de una misma franquicia o aparecen en las recomendaciones. Adicionalmente se obtienen detalles de los miembros del staff. 

In [36]:
def extract_relations(seed_ids, checkpoint=CHPT_EXPANSION, check=25, max_animes=None, max_recs=20,
                      type=None, filter=None, rating=None, sfw=False, traverse_filtered=True):
    """A partir de una lista de IDs de anime, descarga sus datos completos, relaciones, staff y recomendaciones.
    Se guarda un checkpoint para poder reanudar la extracción si se interrumpe.
    seed_ids:       lista de IDs de anime para iniciar la extracción
    checkpoint:     archivo donde se guarda el progreso
    check:          cada cuántos animes procesados se guarda el checkpoint
    max_animes:     máximo de animes a descargar (None = sin límite)
    max_recs:       máximo de recomendaciones a descargar por anime
    type:           tv, movie, ova, special, ona, music, cm, pv, tv_special
    filter:         airing, upcoming, bypopularity, favorite (por defecto: ranking por score)
    rating:         clasificación de los animes (g, pg, pg-13, r-17, r, rx)
    sfw:            True para excluir contenido explícito
    traverse_filtered: Indica si debe recorrer los animes filtrados 
    """
    check_filters(type, filter, rating)
    filters = {"type": type, "filter": filter, "rating": rating, "sfw": sfw}
    seed_ids = [int(i) for i in seed_ids]

    # Cargar checkpoint si existe
    if checkpoint.exists():
        s = json.loads(checkpoint.read_text(encoding="utf-8"))
        anime_list = {int(k): v for k, v in s["anime_list"].items()}
        not_found = set(s["not_found"])
        excluded = set(s.get("excluded", []))
        queue = deque(s["queue"] + s["failed"]) # los que fallaron se reintentan
        print(f"Reanudando: {len(anime_list)} descargados, {len(queue)} pendientes, {len(not_found)} inexistentes")
    else:
        anime_list, not_found, excluded, queue = {}, set(), set(), deque()
        print(f"Iniciando desde cero con {len(seed_ids)} semillas")

    queued = set(queue)

    # Agregamos animes a la fila si se amplió el top desde la última ve
    def enqueue(mal_id):
        if mal_id not in anime_list and mal_id not in not_found and mal_id not in excluded and mal_id not in queued:
            queue.append(mal_id)
            queued.add(mal_id)

    for i in seed_ids:                                    # por si se amplió el top desde la última vez
        enqueue(i)
    failed = []

    # Función para guardar el checkpoint
    def save():
        save_json(checkpoint, {
            "filtros": filters, "anime_list": anime_list, "queue": list(queue),
            "failed": failed, "not_found": sorted(not_found), "excluded": sorted(excluded),
        })

    # Reparación: animes guardados cuyo staff/recomendaciones fallaron en una corrida anterior
    for mal_id, a in anime_list.items():
        if a.get("staff") is None:
            a["staff"] = get_anime_staff(mal_id)
        if a.get("recommendations") is None:
            a["recommendations"] = get_anime_recommendations(mal_id, max_recs)

    procesados = 0
    pbar = tqdm(desc="Animes", unit="anime")
    try:
        while queue:
            # Detener si se alcanzó el número máximo de animes
            if max_animes and len(anime_list) >= max_animes:
                print(f"Se alcanzó max_animes={max_animes}. Quedan {len(queue)} en cola (se conservan en el checkpoint).")
                break
            mal_id = queue.popleft()
            queued.discard(mal_id)
            if mal_id in anime_list or mal_id in not_found:
                continue

            # Obtener datos completos de anime
            data = get_anime_full(mal_id)
            if data is None:                 # error de red: se reintenta en la próxima ejecución
                failed.append(mal_id)
                continue
            if not data:                     # 404
                not_found.add(mal_id)
                continue

            if not meets_filters(data, **filters):
                excluded.add(mal_id)         # no se piden staff ni recomendaciones
                if traverse_filtered:
                    for r in data["relations"]:
                        enqueue(r["mal_id"])
            else:
                data["staff"] = get_anime_staff(mal_id)
                data["recommendations"] = get_anime_recommendations(mal_id, max_recs)
                anime_list[mal_id] = data
                for r in data["relations"]:
                    enqueue(r["mal_id"])

            # Barra de progreso
            procesados += 1
            pbar.update(1)
            pbar.set_postfix(pendientes=len(queue), fallidos=len(failed))
            if procesados % check == 0:
                save()
    finally:
        save()                        
        pbar.close()

    if failed:
        print(f"{len(failed)} animes fallaron por red; vuelve a ejecutar esta celda para reintentarlos.")
    return anime_list

extract_relations([1, 20, 21], max_animes=100, max_recs=20)  # Test de extracción de relaciones

Reanudando: 5445 descargados, 0 pendientes, 0 inexistentes


Animes: 0anime [00:00, ?anime/s]

{1: {'mal_id': 1,
  'mal_url': 'https://myanimelist.net/anime/1/Cowboy_Bebop',
  'title': 'Cowboy Bebop',
  'title_english': 'Cowboy Bebop',
  'title_japanese': 'カウボーイビバップ',
  'type': 'TV',
  'source': 'Original',
  'status': 'Finished Airing',
  'episodes': 26,
  'duration': '24 min per ep',
  'duration_min': 24,
  'rating': 'R - 17+ (violence & profanity)',
  'score': 8.75,
  'scored_by': 1080904,
  'score_rank': 51,
  'popularity_rank': 41,
  'members': 2096866,
  'favorites': 91263,
  'year': 1998,
  'season': 'spring',
  'aired_from': '1998-04-03',
  'synopsis': "Crime is timeless. By the year 2071, humanity has expanded across the galaxy, filling the surface of other planets with settlements like those on Earth. These new societies are plagued by murder, drug use, and theft, and intergalactic outlaws are hunted by a growing number of tough bounty hunters.\n\nSpike Spiegel and Jet Black pursue criminals throughout space to make a humble living. Beneath his goofy and aloof demeanor

**Construir DataFrames**

Obtenemos los datos extraidos en formato JSON y los pasamos a DataFrames

In [37]:
ANIME_COLS = [
    "title", "title_english", "title_japanese", "type", "source", "status", "episodes", "duration",
    "duration_min", "rating", "score", "scored_by", "score_rank", "popularity_rank", "members",
    "favorites", "year", "season", "aired_from", "synopsis", "background", "image_url", "mal_url",
]


def build_dataframes(anime_list, top_ids):
    """Función para construir DataFrames a partir de los JSON
    anime_list: La lista con los datos de los animes
    top_ids: IDs de los top N animes iniciales"""
    top_ids = set(top_ids)
    ids = set(anime_list)
    anime = []
    genre, demographic, theme, producer, person = {}, {}, {}, {}, {}
    a_genre, a_demo, a_theme, a_prod, a_staff, a_rel, a_rec = [], [], [], [], [], [], []

    for mid, a in anime_list.items():
        anime.append({"anime_id": mid, **{c: a.get(c) for c in ANIME_COLS}, "is_top": mid in top_ids})

        for g in a.get("genres", []):
            genre[g["mal_id"]] = (g["name"], False)
            a_genre.append((mid, g["mal_id"]))
        for g in a.get("explicit_genres", []):
            genre[g["mal_id"]] = (g["name"], True)
            a_genre.append((mid, g["mal_id"]))
        for d in a.get("demographics", []):
            demographic[d["mal_id"]] = d["name"]
            a_demo.append((mid, d["mal_id"]))
        for t in a.get("themes", []):
            theme[t["mal_id"]] = t["name"]
            a_theme.append((mid, t["mal_id"]))

        for role, key in (("studio", "studios"), ("producer", "producers"), ("licensor", "licensors")):
            for p in a.get(key, []):
                producer[p["mal_id"]] = p["name"]
                a_prod.append((mid, p["mal_id"], role))

        for s in a.get("staff") or []:
            person[s["person_id"]] = s["name"]
            for pos in s["positions"] or ["N/A"]:
                a_staff.append((mid, s["person_id"], pos.strip()))

        for r in a.get("relations") or []:
            if r["mal_id"] in ids and r["mal_id"] != mid:          
                a_rel.append((mid, r["mal_id"], r["relation"]))

        for r in a.get("recommendations") or []:
            if r["mal_id"] in ids and r["mal_id"] != mid:
                a_rec.append((mid, r["mal_id"], r["votes"]))

    def df(rows, cols):
        return pd.DataFrame(rows, columns=cols).drop_duplicates().reset_index(drop=True)

    dfs = {
        "anime": pd.DataFrame(anime),
        "genre": df([(k, v[0], v[1]) for k, v in genre.items()], ["genre_id", "name", "is_explicit"]),
        "demographic": df(list(demographic.items()), ["demographic_id", "name"]),
        "theme": df(list(theme.items()), ["theme_id", "name"]),
        "producer": df(list(producer.items()), ["producer_id", "name"]),
        "person": df(list(person.items()), ["person_id", "name"]),
        "anime_genre": df(a_genre, ["anime_id", "genre_id"]),
        "anime_demographic": df(a_demo, ["anime_id", "demographic_id"]),
        "anime_theme": df(a_theme, ["anime_id", "theme_id"]),
        "anime_producer": df(a_prod, ["anime_id", "producer_id", "role"]),
        "anime_staff": df(a_staff, ["anime_id", "person_id", "position"]),
        "anime_relation": df(a_rel, ["anime_id", "related_anime_id", "relation_type"]),
        "anime_recommendation": df(a_rec, ["anime_id", "recommended_anime_id", "votes"]),
    }
    return dfs


def save_csvs(dfs, folder=DATA):
    """Almacena los DataFrames en formato .csv
    dfs: DataFrames con la información por guardar
    folder: Carpeta en la que se guardan los DataFrames"""
    for name, d in dfs.items():
        d.to_csv(folder / f"{name}.csv", index=False, encoding="utf-8")
        print(f"{name:22s} {len(d):>8,d} filas  ->  data/{name}.csv")


In [38]:
TOP_N = 1000           
MAX_ANIMES = None    

FILTERS = dict(type=None, filter=None, rating=None, sfw=True)

animes = get_top_anime(amount=TOP_N, **FILTERS)       
top_ids = [a["mal_id"] for a in animes]
print(f"{len(top_ids)} animes en el top")
pd.DataFrame(animes)[["mal_id", "title", "type", "score", "members"]].head()


El checkpoint del top usa otros parámetros: se ignora (bórralo si quieres empezar de cero).


Top anime:   0%|          | 0/40 [00:00<?, ?it/s]

1000 animes en el top


,mal_id,title,type,score,members
0,52991,Sousou no Frieren,TV,9.24,1542421
1,5114,Fullmetal Alchemist: Brotherhood,TV,9.11,3754790
2,9253,Steins;Gate,TV,9.07,2866100
3,38524,Shingeki no Kyojin Season 3 Part 2,TV,9.06,2656795
4,61316,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,TV,9.06,369630


In [39]:
anime_list = extract_relations(top_ids, max_animes=MAX_ANIMES, **FILTERS)
print(f"Total tras expansión: {len(anime_list)} animes "
      f"({len(anime_list) - len(top_ids)} agregados por relaciones)")

Reanudando: 5445 descargados, 0 pendientes, 0 inexistentes


Animes: 0anime [00:00, ?anime/s]

Total tras expansión: 5445 animes (4445 agregados por relaciones)


In [40]:
dfs = build_dataframes(anime_list, top_ids)
save_csvs(dfs)

anime                     5,445 filas  ->  data/anime.csv
genre                        21 filas  ->  data/genre.csv
demographic                   5 filas  ->  data/demographic.csv
theme                        52 filas  ->  data/theme.csv
producer                  1,028 filas  ->  data/producer.csv
person                   19,377 filas  ->  data/person.csv
anime_genre               9,749 filas  ->  data/anime_genre.csv
anime_demographic         2,158 filas  ->  data/anime_demographic.csv
anime_theme               6,634 filas  ->  data/anime_theme.csv
anime_producer           15,923 filas  ->  data/anime_producer.csv
anime_staff             234,608 filas  ->  data/anime_staff.csv
anime_relation           12,769 filas  ->  data/anime_relation.csv
anime_recommendation     10,522 filas  ->  data/anime_recommendation.csv


In [41]:
# Revisión rápida
dfs["anime"][["anime_id", "title", "type", "year", "score", "members", "is_top"]].sort_values("title", ascending=True).head(50)

,anime_id,title,type,year,score,members,is_top
1989,51478,!NVADE SHOW!,Music,2020.0,5.84,1151,False
1111,41060,"""Inochi Sketch"" Movie Jabow x Mameshiba Collab CM",CM,2019.0,5.09,423,False
1090,35632,"""Kinako"" Movie x Mameshiba",Special,2010.0,4.97,804,False
1093,37843,"""Kiss Dekiru Gyouza"" Movie x Mameshiba",CM,2018.0,4.54,815,False
1786,60669,"""Mahouka Koukou no Rettousei"" Yori Shinnen no ...",PV,2025.0,5.55,721,False
4137,58288,"""Pokemon Shirt Sizing"" Concept Movie",PV,2022.0,5.65,724,False
1091,37626,"""R100"" x Mameshiba Original Manners",ONA,2013.0,4.71,866,False
3319,40554,"""Uchuu Senkan Yamato"" to Iu Jidai: Seireki 220...",Movie,2021.0,7.21,4965,False
568,51481,#Aqua Iro Palette,Music,2020.0,6.63,1186,False
4307,62292,#Mugen ni Tomaranai,CM,2023.0,NaN,248,False
